# 03 · Diversity-Aware Re-ranking (Novelty, part 1) — The Engagement Trap

An accuracy-optimised recommender tends to fill every list with the same popular, similar movies. Here we keep the **SVD** model's scores but change *which* 10 movies are shown, and measure what that costs in accuracy and gains in diversity.

**Re-rankers** (both start from the SVD model's top-100 unseen movies for each user):
- **MMR (Maximal Marginal Relevance).** Build the list one movie at a time, each time picking the movie that maximises
  `λ · relevance − (1 − λ) · max similarity to movies already picked`.
  λ = 1 is the plain SVD list; smaller λ means more diversity. Similarity = cosine between SVD movie embeddings.
- **Cluster cap.** Cluster the movies, then walk down the ranked list and allow at most *cap* movies from any one cluster.

**Clustering (Exp 10 – K-Means vs DBSCAN)** on the SVD movie embeddings (V·Σ):
- **K-Means** (on L2-normalised embeddings, k chosen by silhouette score) puts *every* movie in some cluster.
- **DBSCAN** (cosine distance, ε chosen by silhouette) finds dense groups and labels the rest as **noise** (−1). Noise movies are never capped.

**Fair evaluation.** λ and cap are chosen on the *validation* split with a fixed rule: the most diverse setting (highest ILD) that keeps ≥ 95% of plain SVD's validation NDCG@10. Diversity (ILD) is measured with **genre** vectors, which MMR does *not* use (it uses SVD embeddings), so MMR is not directly optimising the metric it is judged on.

In [ ]:
import os, sys, glob, json, time
# On Kaggle, etlib.py comes from an attached dataset; locally it sits next to this notebook.
for hit in glob.glob("/kaggle/input/**/etlib.py", recursive=True):
    sys.path.insert(0, os.path.dirname(hit))
    break
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import etlib as et

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
OUT = et.output_dir()
DATA_DIR = et.find_data_dir()
ratings, items, GENRES = et.load_movielens(DATA_DIR)  # movies with >= 5 ratings
G = et.genre_matrix(items, GENRES)
print("data:", DATA_DIR, "| outputs:", OUT)
print(len(ratings), "ratings |", ratings.user.nunique(), "users |", ratings.item.nunique(), "movies")


def save_fig(fig, name):
    fig.savefig(f"{OUT}/figures/{name}.png", dpi=150, bbox_inches="tight")

## 1. Configuration

In [ ]:
SEEDS = [0, 1, 2]
K = 10
POOL = 100                       # candidates per user that the re-rankers may reorder
SVD_GRID = [5, 10, 20, 40, 60]   # same grid as notebook 02
LAMBDAS = [1.0, 0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3]
CAPS = [1, 2, 3]
TOLERANCE = 0.95                 # keep >= 95% of plain-SVD validation NDCG@10

## 2. K-Means vs DBSCAN on SVD movie embeddings (seed 0)

Before using clusters for re-ranking, look at what each algorithm finds. Embeddings come from an SVD (k = 20) fitted on the seed-0 training split.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score

train0, _ = et.split_per_user(ratings, 0.2, seed=0)
pop0 = et.item_counts(train0)
vecs0 = et.SVDRec(k=20, seed=0).fit(et.to_csr(train0)).item_vectors()
km0, km_table = et.choose_kmeans(vecs0)
db0, db_table, eps0 = et.choose_dbscan(vecs0)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(km_table["k"], km_table["silhouette"], marker="o")
axes[0].set(xlabel="k", ylabel="silhouette (cosine)", title="K-Means: choosing k")
axes[1].plot(db_table["eps"], db_table["n_clusters"], marker="o", label="clusters")
ax2 = axes[1].twinx()
ax2.plot(db_table["eps"], db_table["noise_frac"], marker="s", color="C3", label="noise share")
axes[1].axvline(eps0, ls="--", c="gray")
axes[1].set(xlabel="eps (cosine distance)", ylabel="number of clusters", title="DBSCAN: choosing eps")
ax2.set_ylabel("share of movies labelled noise")
fig.tight_layout()
save_fig(fig, "03_cluster_selection")
plt.show()
km_table.to_csv(f"{OUT}/results/03_kmeans_selection_seed0.csv", index=False)
db_table.to_csv(f"{OUT}/results/03_dbscan_selection_seed0.csv", index=False)
display(km_table.round(4))
display(db_table.round(4))

In [ ]:
xy = PCA(n_components=2, random_state=0).fit_transform(et.normalize_rows(vecs0))
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
axes[0].scatter(xy[:, 0], xy[:, 1], c=km0, cmap="tab20", s=8)
axes[0].set_title(f"K-Means ({len(set(km0))} clusters)")
noise = db0 == -1
axes[1].scatter(xy[noise, 0], xy[noise, 1], c="lightgray", s=6, label="noise")
axes[1].scatter(xy[~noise, 0], xy[~noise, 1], c=db0[~noise], cmap="tab20", s=8)
axes[1].set_title(f"DBSCAN (eps={eps0}, {len(set(db0[~noise]))} clusters, {noise.mean():.0%} noise)")
axes[1].legend()
for ax in axes:
    ax.set(xlabel="PC1", ylabel="PC2")
fig.suptitle("SVD movie embeddings projected to 2-D (colour = cluster)")
save_fig(fig, "03_clusters_2d")
plt.show()

clustered = ~noise
print("Adjusted Rand index K-Means vs DBSCAN (non-noise movies):",
      round(adjusted_rand_score(km0[clustered], db0[clustered]), 3) if clustered.any() else "n/a")
print("Median popularity of DBSCAN noise movies    :", np.median(pop0[noise]) if noise.any() else "n/a")
print("Median popularity of DBSCAN clustered movies:", np.median(pop0[clustered]) if clustered.any() else "n/a")

What is in each K-Means cluster? (dominant genres and the most-rated titles)

In [ ]:
def describe(labels):
    rows = []
    for c in sorted(set(labels)):
        members = np.where(labels == c)[0]
        genre_share = G[members].mean(axis=0)
        top_genres = [GENRES[g] for g in np.argsort(-genre_share)[:3]]
        top_titles = items.title.to_numpy()[members[np.argsort(-pop0[members])[:3]]]
        rows.append({"cluster": c, "size": len(members), "median_pop": np.median(pop0[members]),
                     "top_genres": ", ".join(top_genres), "most_rated": " | ".join(top_titles)})
    return pd.DataFrame(rows).set_index("cluster")

pd.set_option("display.max_colwidth", 120)
display(describe(km0))
describe(db0)

Things to note: does DBSCAN find one dominant cluster plus noise (common when points are spread fairly evenly), or several meaningful groups? Are the noise movies mostly rarely-rated ones (their embeddings are estimated from very few ratings)? Do K-Means clusters line up with genres or eras?

## 3. Choose λ and cap on validation, evaluate on test

For each seed: tune SVD's k (as in notebook 02) → build plain, MMR and cluster-cap lists on validation → pick λ and cap with the 95% rule → refit on full train → evaluate every variant on test once. All variants are reported on test so we can draw the full trade-off curve; the **chosen** ones are marked.

In [ ]:
def tune_svd_k(fit, val, R_fit, seed):
    scores = {k: et.evaluate(et.SVDRec(k, seed).fit(R_fit), fit, val, G, K)["NDCG@10"] for k in SVD_GRID}
    return max(scores, key=scores.get)


def diversity_variants(svd, train_df, seed):
    """Top-K lists for plain SVD, every MMR lambda and every cluster cap."""
    pool, pool_scores = et.candidate_pool(svd.predict_all(), et.seen_mask(train_df), POOL)
    vecs = svd.item_vectors()
    vecs_norm = et.normalize_rows(vecs)
    km, _ = et.choose_kmeans(vecs, seed=seed)
    db, _, _ = et.choose_dbscan(vecs)
    lists = {("SVD", None): pool[:, :K]}
    for lam in LAMBDAS:
        lists[("MMR", lam)] = et.rerank_all(pool, pool_scores, "mmr", K, item_vecs_norm=vecs_norm, lam=lam)
    for cap in CAPS:
        lists[("KMeans-cap", cap)] = et.rerank_all(pool, pool_scores, "cap", K, labels=km, cap=cap)
        lists[("DBSCAN-cap", cap)] = et.rerank_all(pool, pool_scores, "cap", K, labels=db, cap=cap)
    return lists


def pick(rows, method, base_ndcg):
    """Most diverse setting (highest ILD) that keeps >= TOLERANCE of plain-SVD NDCG@10."""
    ok = [r for r in rows if r["method"] == method and r["NDCG@10"] >= TOLERANCE * base_ndcg]
    return max(ok, key=lambda r: r["ILD"])["param"] if ok else None


val_rows, test_rows, chosen = [], [], []
start = time.time()
for seed in SEEDS:
    train, test = et.split_per_user(ratings, 0.2, seed)
    fit, val = et.split_per_user(train, 0.2, seed + 100)
    R_fit, R_train = et.to_csr(fit), et.to_csr(train)
    svd_k = tune_svd_k(fit, val, R_fit, seed)

    # validation: choose lambda and caps
    lists = diversity_variants(et.SVDRec(svd_k, seed).fit(R_fit), fit, seed)
    rows = [{"method": m, "param": p, "seed": seed, **et.evaluate_lists(top, fit, val, G)}
            for (m, p), top in lists.items()]
    val_rows += rows
    base = rows[0]["NDCG@10"]
    choice = {"seed": seed, "svd_k": svd_k,
              "MMR": pick(rows, "MMR", base) or 1.0,
              "KMeans-cap": pick(rows, "KMeans-cap", base) or max(CAPS),
              "DBSCAN-cap": pick(rows, "DBSCAN-cap", base) or max(CAPS)}
    chosen.append(choice)

    # test: refit on full train, evaluate every variant once
    lists = diversity_variants(et.SVDRec(svd_k, seed).fit(R_train), train, seed)
    for (m, p), top in lists.items():
        test_rows.append({"method": m, "param": p, "seed": seed, "chosen": p == choice.get(m, p),
                          **et.evaluate_lists(top, train, test, G)})
    pop_top = et.recommend(et.Popularity().fit(R_train).rank_all(), train, K)
    test_rows.append({"method": "Popularity (reference)", "param": None, "seed": seed, "chosen": True,
                      **et.evaluate_lists(pop_top, train, test, G)})
    print(f"seed {seed} | choice {choice} | {time.time() - start:.0f}s")

chosen = pd.DataFrame(chosen)
chosen

## 4. Results: chosen settings on test (mean ± std over 3 seeds)

In [ ]:
val_df, test_df = pd.DataFrame(val_rows), pd.DataFrame(test_rows)
val_df.to_csv(f"{OUT}/results/03_rerank_validation.csv", index=False)
test_df.to_csv(f"{OUT}/results/03_rerank_test_all_variants.csv", index=False)
chosen.to_csv(f"{OUT}/results/03_chosen_settings.csv", index=False)

final = test_df[test_df["chosen"]].rename(columns={"method": "model"})
table, agg = et.mean_std_table(final.drop(columns=["param", "chosen"]).to_dict("records"))
table.to_csv(f"{OUT}/results/03_rerank_mean_std.csv")
table

## 5. The accuracy–diversity trade-off (test, mean over seeds)

Each point is one setting. The MMR line goes from λ = 1 (plain SVD) towards more diversity. These are *descriptive* curves on test; the operating points were chosen on validation above.

In [ ]:
curve = test_df.groupby(["method", "param"], dropna=False, sort=False)[
    ["NDCG@10", "ILD", "Coverage", "AvgPop", "Gini"]].mean().reset_index()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, x in zip(axes, ["ILD", "Coverage", "AvgPop"]):
    mmr_c = curve[curve.method == "MMR"]
    ax.plot(mmr_c[x], mmr_c["NDCG@10"], "-o", label="MMR (λ sweep)")
    for lam, xv, yv in zip(mmr_c.param, mmr_c[x], mmr_c["NDCG@10"]):
        ax.annotate(f"λ={lam}", (xv, yv), textcoords="offset points", xytext=(4, 4), fontsize=8)
    for method, marker in [("KMeans-cap", "s"), ("DBSCAN-cap", "^")]:
        c = curve[curve.method == method]
        ax.plot(c[x], c["NDCG@10"], marker, ms=8, label=method + " (cap 1/2/3)")
        for cap, xv, yv in zip(c.param, c[x], c["NDCG@10"]):
            ax.annotate(f"cap={int(cap)}", (xv, yv), textcoords="offset points", xytext=(4, -10), fontsize=8)
    ref = curve[curve.method == "Popularity (reference)"]
    ax.plot(ref[x], ref["NDCG@10"], "kx", ms=10, label="Popularity (reference)")
    ax.set(xlabel=x, ylabel="NDCG@10")
axes[0].legend(fontsize=8)
axes[0].set_title("Accuracy vs genre diversity (ILD ↑)")
axes[1].set_title("Accuracy vs catalogue coverage (↑)")
axes[2].set_title("Accuracy vs popularity of recommendations (↓ = less blockbuster-heavy)")
fig.tight_layout()
save_fig(fig, "03_tradeoff")
plt.show()
curve.round(4)

**Things to look at**
- How much NDCG@10 does each re-ranker give up, relative to the std across seeds? A drop smaller than the seed-to-seed noise is effectively free.
- How much do ILD and Coverage go up, and does AvgPop go down?
- K-Means cap vs DBSCAN cap: if DBSCAN marks most movies as noise, its cap barely changes the list — that is a real finding about DBSCAN on this kind of data, not a bug.

## 6. Export for the demo site

The demo needs SVD movie embeddings and cluster labels. For the demo only, SVD is refitted on **all** ratings (no evaluation happens on this model), with the k, λ and caps chosen most often across seeds above. The demo builds a new user's taste vector in the browser from the movies they pick ("fold-in").

In [ ]:
def most_common(col):
    return pd.Series(chosen[col]).mode()[0]

k_demo = int(most_common("svd_k"))
svd_all = et.SVDRec(k_demo, seed=0).fit(et.to_csr(ratings))
vecs_all = svd_all.item_vectors()
km_all, _ = et.choose_kmeans(vecs_all)
db_all, _, eps_all = et.choose_dbscan(vecs_all)

et.save_json({"k": k_demo, "item_vectors": np.round(vecs_all, 4).tolist()},
             f"{OUT}/demo_data/item_factors.json")
et.save_json({"kmeans": km_all.tolist(), "dbscan": db_all.tolist(), "dbscan_eps": eps_all},
             f"{OUT}/demo_data/clusters.json")
et.save_json({
    "lambda": float(most_common("MMR")),
    "cap": {"kmeans": int(most_common("KMeans-cap")), "dbscan": int(most_common("DBSCAN-cap"))},
    "table": {n: {m: {"mean": et.json_num(agg[m]["mean"][n]), "std": et.json_num(agg[m]["std"][n])}
                  for m in ["Precision@10", "Recall@10", "NDCG@10", "Coverage", "Gini", "ILD", "AvgPop"]}
              for n in agg.index},
    "curve": [{"method": r["method"], "param": None if pd.isna(r["param"]) else float(r["param"]),
               **{m: et.json_num(r[m]) for m in ["NDCG@10", "ILD", "Coverage", "AvgPop", "Gini"]}}
              for r in curve.to_dict("records")],
}, f"{OUT}/demo_data/rerank.json")
print("demo k =", k_demo, "| K-Means clusters:", len(set(km_all)), "| DBSCAN clusters:",
      len(set(db_all) - {-1}), "noise:", f"{(db_all == -1).mean():.0%}")
print("total time", round(time.time() - start), "s")